In [132]:
import pandas as pd
import json
import random

pd.set_option("display.max_colwidth", None)   # 컬럼 텍스트 안 자름
pd.set_option("display.max_columns", None)    # 컬럼 개수 제한 해제
pd.set_option("display.width", None)          # 출력 폭 제한 해제

### 01) 멀티에이전트 Excel `Success 시트` → `csv` 형식

- `result`는 JSON 문자열(또는 이미 list)로 된 **트리플 배열**이어야 함. 각 원소는 `head`, `relation`, `tail` 키(멀티에이전트 CSV와 동일).
- 한 행(한 책)이 여러 트리플이면 **트리플 개수만큼 행으로 확장**하고, `title` 컬럼에는 그 행의 책 URI(`item` 또는 `title`)를 반복해 넣음.
- `csv` 4컬럼: `head_id:token`, `relation_id:token`, `tail_id:token`, `title`.

`results_from_multiagents_books_all.xlsx` 경로는 노트북 실행 위치에 맞게 수정하세요.

In [ ]:
def json_to_triples_list(
    df,
    #out_path: str,
    item_cols=("item", "title"),
    result_col: str = "result",
):
    """Success 시트 DataFrame → csv 동일 스키마."""
    item_col = next((c for c in item_cols if c in df.columns), None)
    if item_col is None:
        raise ValueError(f"컬럼 필요: {item_cols} 중 하나. 실제: {list(df.columns)}")
    if result_col not in df.columns:
        raise ValueError(f"'{result_col}' 컬럼 없음")

    rows_out = []
    for _, row in df.iterrows():
        book_id = str(row[item_col]).strip()
        raw = row[result_col]
        if pd.isna(raw) or (isinstance(raw, str) and not raw.strip()):
            continue
        if isinstance(raw, str):
            try:
                triples = json.loads(raw)
            except json.JSONDecodeError:
                continue
        elif isinstance(raw, list):
            triples = raw
        else:
            continue
        if not isinstance(triples, list):
            continue
        for t in triples:
            if not isinstance(t, dict):
                continue
            h = t.get("head")
            r = t.get("relation")
            tail = t.get("tail")
            if h is None or r is None or tail is None:
                continue
            rows_out.append(
                {
                    "head_id:token": str(h).strip(),
                    "relation_id:token": str(r).strip(),
                    "tail_id:token": str(tail).strip(),
                    "title": book_id,
                }
            )

    out_df = pd.DataFrame(rows_out)
    #out_df.to_csv(out_path, index=False, encoding="utf-8")
    #print(f"시트 행 {len(df)} → 트리플 행 {len(out_df)} → {out_path}")
    return out_df




In [ ]:

#
movies_success = pd.read_excel("./멀티에이전트로 생성한 KG 파일.xlsx", sheet_name="Success")
print(len(movies_success))
movies_success.tail(1)




In [ ]:
# 예: 엑셀에서 읽기 (header=1 이 맞는지 시트 열어서 확인)
kg_from_success_movies = json_to_triples_list(movies_success)

In [ ]:
#원본 전체에서 랜덤으로 50% 아이템만 필터 

#half = "사전에 정제한 50% 아이템 리스트 파일"
#original = 원본 전체 데이터

random_half_items = half.sample(n=3822, random_state=42).reset_index(drop=True)
random_half_items.head(5)

org_filter_half = origianl.merge(half, left_on='head_id:token', right_on='title', how='inner')
print(len(org_filter_half))
org_filter_half.tail(5)

#org_filter_half.to_csv("org_filter_half.csv", index=False)

75606


### 02) Cleasing 된 데이터를 Test Case(비율)에 따라 생성

In [50]:
import numpy as np
import pandas as pd


def build_hybrid_kg_by_title_fraction(
    path_original: str,
    path_semi: str,
    pct: int,
    #domain: str,
    #out_prefix: str = "kg_hybrid",
    title_col: str = "title",
    #fractions_pct: tuple = (10, 20, 30, 40, 50, 60, 70, 80, 90),
    random_state: int = 42,
    cumulative: bool = False,
):
    df_o = pd.read_csv(path_original)
    #df_o = original
    df_s = pd.read_csv(path_semi)
    #df_s = semi
    pct = pct

    common = sorted(set(df_o[title_col]) & set(df_s[title_col]))
    N = len(common)
    if N == 0:
        raise ValueError("original과 semi에 공통 title 없음")

    rng = np.random.default_rng(random_state)
    if cumulative:
        order = common.copy()
        rng.shuffle(order)

    #for pct in fractions_pct:
    k = int(round(N * pct / 100))
    k = max(0, min(k, N))

    if k == 0:
        chosen = set()
    elif cumulative:
        chosen = set(order[:k])
    else:
        chosen = set(rng.choice(common, size=k, replace=False))

    # 같은 title은 한 소스에서만 (전부 orig 또는 전부 semi)
    part_o = df_o[~df_o[title_col].isin(chosen)]
    part_s = df_s[df_s[title_col].isin(chosen)]
    hybrid = pd.concat([part_o, part_s], ignore_index=True)

    #out_path = f"{out_prefix}_semi{pct}pct.csv"
    #hybrid.to_csv(out_path, index=False, encoding="utf-8")
    #print(f"{pct}%  semi title {k}/{N}  ->  {out_path}  rows={len(hybrid)}")

    return N, common, hybrid




In [ ]:
cnt = "half"
domain = "movies" #books

#semi = f'kg_from_success_{domain}_dbo'
#semi = f'triples_standardized_movies_domain'
semi = f'triples_standardized_movies_joint' # 멀티 에이전트로 생성한 kg success 
original = f'{domain}_half_filtered_kg_random_{cnt}' #org_filter_half

original_path = f'./{original}.csv'
semi_path = f'./{semi}.csv'
print(original_path)
print(semi_path)

./movies_half_filtered_kg_random_half.csv
./triples_standardized_movies_joint.csv


In [ ]:
for pct in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100] :
#for pct in [100] :

     N, common,hybrid = build_hybrid_kg_by_title_fraction(
                    original_path,
                    semi_path,
                    pct=pct)

     hybrid.to_csv(f"./경로/kg_{cnt}_hybrid_semi{pct}pct_{domain}.csv", index=False)